In [1]:
# Hämta pandas och ge det kortnamnet pd.
import pandas as pd

# Läs CSV-filen och spara tabellen i df.
# ../ betyder: gå upp en mapp, sedan in i data/.

df = pd.read_csv("../data/ecommerce_sales.csv")

# Läs filen ecommerce_sales.csv från datamappen och kalla tabellen df.
# pd är kortnamnet för pandas.
#read_csv(...) läser en CSV-fil och skapar en tabell.
#"../data/ecommerce_sales.csv" berättar var filen finns:- ../ → gå upp en mapp.
# data/ → gå in i mappen data.
# ecommerce_sales.csv → välj filen.

# Visa de fem första raderna i tabellen.
df.head()

#df = ger den inlästa tabellen namnet df, så att vi kan använda den senare.


,order_id,date,city,category,price,units,revenue
0,100000,2024-04-19,Stockholm,Sports,530.95,4,2123.80
1,100001,2024-05-06,Malmö,Sports,653.28,3,1959.84
2,100002,2024-03-07,Uppsala,Clothing,177.98,4,711.92
3,100003,2024-04-08,Göteborg,Home,348.57,3,1045.71
4,100004,2024-01-18,Uppsala,Electronics,777.07,3,2331.21


1. Inläsning och datakontroll

#Syftet är att läsa in ecommerce_sales.csv och kontrollera datakvaliteten.


In [2]:
df.shape

#df är vår inlästa tabell.
# shape visar antalet rader och kolumner, i den ordningen.

(2500, 7)

## Observation: tabellens storlek

Den inlästa tabellen har 2 500 rader och 7 kolumner.
Vi har ännu inte kontrollerat om varje rad motsvarar en unik order.
Inga värden har ändrats i detta steg.

In [3]:
df.info()

# df är vår tabell.
#.info() visar en översikt över kolumnerna, deras datatyper och antalet ifyllda värden.
# Non-Null Count betyder antal värden som inte saknas. 
# Eftersom tabellen har 2 500 rader kan ett lägre antal visa att värden saknas.

<class 'pandas.DataFrame'>
RangeIndex: 2500 entries, 0 to 2499
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   order_id  2500 non-null   int64  
 1   date      2500 non-null   str    
 2   city      2500 non-null   str    
 3   category  2500 non-null   str    
 4   price     2500 non-null   float64
 5   units     2500 non-null   int64  
 6   revenue   2500 non-null   float64
dtypes: float64(2), int64(2), str(3)
memory usage: 136.8 KB


## Observation: datatyper och saknade värden

Alla sju kolumner har 2 500 ifyllda värden enligt df.info().
Pandas visar inga saknade värden.
Kolumnen date är lagrad som text och behöver kontrolleras som datum.
Vi har ännu inte kontrollerat om värdena är rimliga.
Inga data har ändrats.

In [4]:
df[["price", "units", "revenue"]].describe()

# df[["price", "units", "revenue"]] väljer de tre sifferkolumnerna.
# describe() sammanfattar deras värden, exempelvis minsta värde, medelvärde och största värde.

,price,units,revenue
count,2500.000000,2500.000000,2500.000000
mean,476.213212,2.985200,1401.106892
std,312.968413,1.429755,1166.944967
min,50.000000,1.000000,50.000000
25%,254.960000,2.000000,577.960000
50%,413.045000,3.000000,1088.690000
75%,624.427500,4.000000,1884.750000
max,2092.530000,9.000000,9192.600000


## Observation: numeriska värden

Price, units och revenue innehåller inga nollor eller negativa värden.
Högsta priset är 2092,53 och högsta antalet enheter är 9.
Högsta intäkten per rad är 9192,60.

Fråga till fortsatt kontroll:
Är de högsta värdena rimliga, och stämmer revenue med price gånger units?

Inga data har ändrats.

På raden max (största värdet) står:
- price: 2092.53 → högsta priset.
- units: 9 → högsta antalet enheter på en rad.
- revenue: 9192.60 → högsta intäkten på en rad.
På raden min (minsta värdet) står:
- price: 50
- units: 1
- revenue: 50
Eftersom de minsta värdena är större än noll, finns inga nollor eller negativa värden i dessa kolumner.
Punkten i 2092.53 är decimaltecken i Python. På svenska skriver vi samma tal som 2092,53.
Frågan om revenue stämmer med price × units är något vi ännu inte har kontrollerat.

In [5]:
kontrollerade_datum = pd.to_datetime(df["date"], errors="coerce")
kontrollerade_datum.isna().sum()

# pd.to_datetime(...) försöker tolka värdena i date som datum.
#errors="coerce" gör att värden som inte kan tolkas blir NaT, vilket betyder saknat datum.
#kontrollerade_datum = sparar resultatet i en separat variabel. df ändras inte.
#.isna().sum() räknar hur många datum som saknas efter försöket.

np.int64(0)

#np.int64(0) är ett resultat
#np står för NumPy, ett bibliotek som pandas använder för att hantera tal.
#int64 betyder att värdet lagras som ett heltal.
#(0) är själva värdet: noll.
#I vår kontroll betyder det:
#0 datumvärden kunde inte tolkas som datum.

Alla datum kunde alltså tolkas. Det säger ännu inget om vilken tidsperiod de ligger inom.

In [6]:
kontrollerade_datum.min(), kontrollerade_datum.max()

#kontrollerade_datum innehåller datumen från vår förra kontroll.
#min() hämtar det tidigaste datumet.
#max() hämtar det senaste datumet.
#Kommatecknet gör att båda resultaten visas.


(Timestamp('2024-01-01 00:00:00'), Timestamp('2024-06-30 00:00:00'))

Resultatet visar att datan sträcker sig från 1 januari 2024 till 30 juni 2024, alltså första halvåret 2024.
## Observation: tidsperiod

Tidigaste datumet är 2024-01-01.
Senaste datumet är 2024-06-30.
Datan omfattar därmed första halvåret 2024.

Fråga till gruppmötet: Är detta den förväntade tidsperioden?

Inga data har ändrats.

In [7]:
# Nu undersöker vi vilka städer som finns och hur många rader varje stad har. 
# Det kan hjälpa oss att upptäcka exempelvis 
# olika stavningar av samma stad. Vi ändrar inga data.

df["city"].value_counts(dropna=False)

# df["city"] väljer kolumnen med städer.
#value_counts() räknar hur många gånger varje stad förekommer.
# dropna=False gör att även saknade värden skulle visas i räkningen.
# Detta räknar rader per stad, inte intäkt eller säkert antal unika order.

city
Stockholm    899
Göteborg     586
Malmö        396
Uppsala      381
Västerås     238
Name: count, dtype: int64

## Observation: städer

Kolumnen city innehåller fem olika stadsnamn:
Stockholm, Göteborg, Malmö, Uppsala och Västerås.

Stockholm har flest rader (899) och Västerås minst (238).
Antalet rader säger inte vilken stad som har högst intäkt.



In [8]:
# vilka produktkategorier som finns.

df["category"].value_counts(dropna=False)

# df["category"] väljer kolumnen med produktkategorier.
#value_counts() räknar antalet rader för varje kategori.
# dropna=False gör att även saknade värden skulle visas.

category
Electronics    618
Clothing       509
Home           490
Sports         415
Beauty         241
Toys           227
Name: count, dtype: int64

## Observation: kategorier

Kolumnen category innehåller sex kategorier:
Electronics, Clothing, Home, Sports, Beauty och Toys.

Inga uppenbara stavningsvarianter syns i listan.
Electronics har flest rader (618) och Toys minst (227).
Intäkt per kategori har ännu inte undersökts.


In [9]:
#  men den kontrolleras om några rader är exakt likadana i alla sju kolumner.
#  Återkommande order-ID är en annan kontroll och betyder inte automatiskt att en rad är fel.

df.duplicated().sum()

# df.duplicated() markerar en rad som True om exakt samma rad redan förekommit.
#.sum() räknar dessa markeringar.

np.int64(0)

## Observation: exakt upprepade rader

Kontrollen df.duplicated().sum() gav 0.
Det finns inga exakt upprepade rader i alla sju kolumner.

Återkommande order-ID har ännu inte kontrollerats.
Inga rader har tagits bort.

In [10]:
#Nu kontrollerar vi om samma order-ID förekommer på flera rader.
#Det är viktigt inför din senare beräkning av genomsnittligt ordervärde,
#AOV. Återkommande ID är inte automatiskt fel.

df["order_id"].duplicated().sum()

# - df["order_id"] väljer bara kolumnen med order-ID.
#.duplicated() markerar förekomster efter den första av samma ID.
# .sum() räknar markeringarna.
#Om ett ID förekommer tre gånger räknas alltså två extra förekomster.
#Vi ändrar inga data.

np.int64(0)

#np.int64(0) betyder att inga order-ID återkommer. Alla 2 500 rader har alltså olika order-ID i den här filen.np.int64(0) betyder att inga order-ID återkommer. Alla 2 500 rader har alltså olika order-ID i den här filen.

## Observation: order-ID

Inga order-ID återkommer i filen.
Tabellen har 2 500 rader med 2 500 unika order-ID.

Fråga till gruppmötet: Motsvarar varje rad en hel order?

Inga data har ändrats.

In [11]:
# Nu kontrollerar vi om intäkten (revenue) stämmer med pris gånger antal (price × units).
#  Det hjälper oss att hitta möjliga räknefel. Vi ändrar inga data

beraknad_intakt = (df["price"] * df["units"]).round(2)
skillnad = (df["revenue"] - beraknad_intakt).abs()
(skillnad > 0.01).sum()

#Rad för rad:
# Rad 1: Multiplicerar pris med antal och avrundar till två decimaler.
#  Resultatet sparas separat i beraknad_intakt.
#Rad 2: Räknar skillnaden mot filens intäkt. .abs() gör skillnaden positiv, 
# så att både för höga och för låga värden upptäcks.
#Rad 3: Räknar rader där skillnaden är större än 0.01.
#  Den lilla toleransen hjälper oss att undvika att obetydliga
#  decimalfel räknas som avvikelser.

np.int64(0)

## Observation: kontroll av intäkt

Revenue jämfördes med price gånger units, avrundat till två decimaler.
Ingen rad hade en absolut skillnad större än 0,01.

Inga data har ändrats.

In [12]:
df.nlargest(5, "revenue")

# Det visar de fem raderna med högst intäkt.
#  5 anger antalet rader och "revenue" kolumnen vi jämför. Inga data ändras.

,order_id,date,city,category,price,units,revenue
1411,101411,2024-02-26,Stockholm,Electronics,1838.52,5,9192.60
1339,101339,2024-02-26,Göteborg,Electronics,1703.37,5,8516.85
1491,101491,2024-06-04,Västerås,Electronics,2092.53,4,8370.12
1733,101733,2024-06-05,Uppsala,Electronics,1227.73,6,7366.38
1518,101518,2024-01-12,Göteborg,Electronics,1403.69,5,7018.45


## Observation: högsta intäkterna

De fem raderna med högst intäkt tillhör kategorin Electronics.
Den högsta intäkten är 9192,60 och kommer från priset 1838,52 gånger 5 enheter.

Höga värden är inte automatiskt fel. Inget uppenbart räknefel syns.
Raderna behålls.

Fråga till gruppmötet: Är dessa priser och antal rimliga för webbutiken?

In [13]:
df = pd.read_csv("../data/ecommerce_sales.csv")
df.head()

# pd.read_csv(...) läser CSV-filen.
#../ betyder att gå upp en mapp från notebooks/.
#data/ecommerce_sales.csv pekar på filen i datamappen.
#df = sparar tabellen i variabeln df.
#df.head() visar de fem första raderna.

,order_id,date,city,category,price,units,revenue
0,100000,2024-04-19,Stockholm,Sports,530.95,4,2123.80
1,100001,2024-05-06,Malmö,Sports,653.28,3,1959.84
2,100002,2024-03-07,Uppsala,Clothing,177.98,4,711.92
3,100003,2024-04-08,Göteborg,Home,348.57,3,1045.71
4,100004,2024-01-18,Uppsala,Electronics,777.07,3,2331.21


In [14]:
def load_and_clean_data(file_path):
    """Läs in CSV-filen utan att ändra några värden."""
    data = pd.read_csv(file_path)
    return data

# Rad för rad:
#  def skapar en funktion med namnet load_and_clean_data.
# file_path är filsökvägen som funktionen ska få när vi använder den.
#  Texten mellan """ beskriver vad funktionen gör just nu.
# data = pd.read_csv(file_path) läser filen och sparar tabellen i data.
# return data lämnar tillbaka tabellen.
# Raderna inuti funktionen är indragna med fyra mellanslag. Det visar att de hör till funktionen.

In [15]:
# Skapa en funktion som tar emot sökvägen till en fil.
def load_and_clean_data(file_path):
    """Läs in CSV-filen och omvandla date till datum."""

    # Läs CSV-filen och spara tabellen i variabeln data.
    data = pd.read_csv(file_path)

    # Omvandla datumkolumnens text till datum.
    # Stoppa med ett felmeddelande om något datum inte kan tolkas.
    data["date"] = pd.to_datetime(data["date"], errors="raise")

    # Lämna tillbaka tabellen till den som anropar funktionen.
    return data

# ”När jag får en filsökväg läser jag filen, 
# omvandlar datumkolumnen och lämnar tillbaka tabellen.”
#-kommentarerna förklarar bara koden.
#  Python kör dem inte. CSV-filen på datorn förblir oförändrad.

In [16]:
# Läs in filen med den uppdaterade funktionen.
test_data = load_and_clean_data("../data/ecommerce_sales.csv")

# Visa datumkolumnens datatyp.
test_data["date"].dtype

# å läser vi koden:
#Första kodraden: Kör funktionen och sparar tabellen i test_data.
#Andra kodraden: Väljer kolumnen date. .dtype visar vilken datatyp den har.

dtype('<M8[us]')

## Funktion: load_and_clean_data

Funktionen tar emot en filsökväg, läser CSV-filen och omvandlar date till datum.
Om ett datum inte kan tolkas stoppas funktionen med ett felmeddelande.

Funktionen har testats:
- Tabellen innehåller 2 500 rader och 7 kolumner.
- Date har datumtyp efter omvandlingen.

Inga rader tas bort eller värden fylls i.
CSV-filen ändras inte.

## Slutkontroll – Sprint 1

Kärnan startades om och alla celler kördes i ordning utan fel.

Datakontrollerna visade:
- 2 500 rader och 7 kolumner.
- Inga saknade värden enligt pandas.
- Inga exakt upprepade rader eller återkommande order-ID.
- Alla datum kunde tolkas och ligger mellan 2024-01-01 och 2024-06-30.
- Inga nollor eller negativa värden i price, units eller revenue.
- Revenue stämmer med price gånger units inom toleransen 0,01.

Beslut:
Datum omvandlas till datumtyp i funktionen.
Inga rader tas bort och inga värden fylls i.
CSV-filen bevaras oförändrad.

Frågor till gruppmötet:
- Motsvarar varje rad en hel order?
- Är tidsperioden och de högsta priserna och antalen förväntade?